# Lesson 4 — From Data Tables to Scientific Plots — Solutions

> **Teacher reference** — This notebook contains worked solutions. Do not share with students.


**Course:** Introductory Python for Materials Science PhD Students
**Session type:** hands-on laboratory (third hands-on session)
**Nominal duration:** 3 hours — realistic core about 2 hours
**Main focus:** plotting, simple data analysis, and a first curve fit

## Learning goals
By the end of this session you should be able to:
- read a data table and build the columns you want to plot;
- make line plots, scatter plots, histograms, bar plots (with error bars) and box plots;
- plot one sample and several samples together;
- compute simple summary metrics per sample;
- **fit a straight line to data** (here: the elastic modulus from a stress-strain curve);
- compare groups of samples and save a figure.

## The thread for today
We keep the tensile-test example from the previous lessons (samples `S01`-`S05`, the file `tensile_raw.csv`). We already found that **quenched is the strongest treatment**. Today we *show* it with plots, we *measure* a material property by **fitting**, and we ask a scientist's follow-up question: **how confident are we in the differences between treatments?**

Some sections are simply a tour of a plot type — that is fine: this lesson is about plotting and analysis, not only the story.

## 0. Setup
Run this cell first. It loads the libraries and sets a couple of plotting defaults.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Plotting defaults - purely cosmetic, you can ignore these two lines
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

print("Setup OK")

## Loading the data
The next cell is **🔧 plumbing**: a small helper that finds the data file and reads it, including inside the browser (JupyterLite). You do **not** need to understand it — in your own work you would simply write `pd.read_csv("data/tensile_raw.csv")`.

In [ ]:
# 🔧 PLUMBING - you do NOT need to understand this cell.
# In your own work you would simply write:
#       df = pd.read_csv("data/tensile_raw.csv")
# This version just tries a few locations so the file always loads,
# including inside the browser (JupyterLite). Run it and move on.
# ----------------------------------------------------------------------

def load_table(name):
    # The normal way: look in the data folder, then the current folder.
    for folder in ("data", "../data", "."):
        try:
            return pd.read_csv(Path(folder) / name)
        except Exception:
            pass
    # Last resort (browser only): download the file from the course website.
    import js
    from pyodide.http import open_url
    base = str(js.location.href).split("/extensions/")[0]
    return pd.read_csv(open_url(f"{base}/files/data/{name}"))


df = load_table("tensile_raw.csv")   # same as pd.read_csv("data/tensile_raw.csv")
df.head()

## 1. Prepare the data: calculated columns
Before plotting we build the quantities we want on the axes. For a tensile test:
```text
area   = pi * (diameter / 2)**2
stress = force / area          # in MPa, because 1 N/mm^2 = 1 MPa
strain = displacement / gauge_length
```
You already did this in the previous lesson, so here we just write the three columns directly.

In [ ]:
df["area_mm2"] = np.pi * (df["diameter_mm"] / 2) ** 2
df["stress_MPa"] = df["force_N"] / df["area_mm2"]
df["strain"] = df["displacement_mm"] / df["gauge_length_mm"]

df[["sample_id", "treatment", "stress_MPa", "strain"]].head()

## 2. First plot: one stress-strain curve
A stress-strain curve is just a line plot: strain on the x-axis, stress on the y-axis. We start with a single sample.

In [ ]:
sample_id = "S03"                      # quenched
sample = df[df["sample_id"] == sample_id]

plt.figure()
plt.plot(sample["strain"], sample["stress_MPa"], marker="o", markersize=3)
plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title(f"Stress-strain curve: {sample_id}")
plt.show()

### Exercise 1 — Change the sample
Copy the code above into the cell below and change `sample_id` to `S01`, `S04`, or `S05`.
- Does the curve have the same shape?
- Does the maximum stress change? And the maximum strain?

In [ ]:
# Exercise 1 — SOLUTION

sample_id = "S01"
sample = df[df["sample_id"] == sample_id]

plt.figure()
plt.plot(sample["strain"], sample["stress_MPa"], marker="o", markersize=3)
plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title(f"Stress-strain curve: {sample_id}")
plt.show()

# S01 (annealed): lower max stress (~354 MPa) but much higher max strain (~0.29)
# The shape is similar (rises, peaks, drops) but the peak is broader


## 3. Several samples in one figure
To compare samples we draw each curve in the same figure, using a `for` loop and a `label` for the legend.

In [ ]:
sample_ids = ["S01", "S03", "S05"]     # annealed, quenched, as_received

plt.figure()
for sample_id in sample_ids:
    sample = df[df["sample_id"] == sample_id]
    plt.plot(sample["strain"], sample["stress_MPa"], label=sample_id)

plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title("Stress-strain curves for several samples")
plt.legend()
plt.show()

### Exercise 2 — Plot all samples
`df["sample_id"].unique()` gives every sample id in the table. Use it in the loop to plot **all** the curves at once.

In [ ]:
# Exercise 2 — SOLUTION

plt.figure()
for sample_id in df["sample_id"].unique():
    sample = df[df["sample_id"] == sample_id]
    plt.plot(sample["strain"], sample["stress_MPa"], label=sample_id)

plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title("Stress-strain curves: all samples")
plt.legend()
plt.show()


## 4. One number per sample: summary metrics
Plots are great, but we also want numbers. For each sample we take the **maximum stress** and the **strain at which it occurs**. `idxmax()` gives the row where a column reaches its maximum.

In [ ]:
rows = []
for sample_id in df["sample_id"].unique():
    sample = df[df["sample_id"] == sample_id]
    top = sample.loc[sample["stress_MPa"].idxmax()]
    rows.append({
        "sample_id": sample_id,
        "treatment": top["treatment"],
        "max_stress_MPa": top["stress_MPa"],
        "strain_at_max": top["strain"],
    })

metrics = pd.DataFrame(rows).sort_values("max_stress_MPa", ascending=False)
metrics

## 5. Scatter plot: strength vs ductility
A scatter plot is the right choice when each point is one object — here, one sample. We put strain-at-max (a rough proxy for ductility) on x and max stress (strength) on y.

In [ ]:
plt.figure()
plt.scatter(metrics["strain_at_max"], metrics["max_stress_MPa"])

# label each point with its sample id
for _, row in metrics.iterrows():
    plt.annotate(row["sample_id"], (row["strain_at_max"], row["max_stress_MPa"]))

plt.xlabel("Strain at maximum stress [-]")
plt.ylabel("Maximum stress [MPa]")
plt.title("Strength vs ductility (one point per sample)")
plt.show()

### Exercise 3 — Read the scatter
Which sample sits highest (strongest)? Which is the most ductile (furthest right)? Are they the same sample?
*(No new code needed — just look and answer in a comment.)*

In [ ]:
# Exercise 3 — SOLUTION (interpretation — no new code needed)

# Strongest (highest on the y-axis): S03 (quenched) — highest max stress (~516 MPa)
# Most ductile (furthest right):     S01 (annealed)  — highest strain at max stress (~0.29)
# They are NOT the same sample:
#   quenching maximises strength but reduces ductility;
#   annealing does the opposite.


## 6. Fitting a line: the elastic modulus
At small strains the stress-strain curve is roughly a **straight line**. Its slope is the **Young's (elastic) modulus** `E`:
```text
stress ≈ E * strain        (small-strain, elastic region)
```
We estimate `E` by fitting a straight line to the low-strain points with `np.polyfit(x, y, 1)`. The `1` means *degree 1* (a straight line); it returns the **slope** and the **intercept** of the best-fit line `y = slope*x + intercept`.

In [ ]:
sample_id = "S03"
sample = df[df["sample_id"] == sample_id]

# the elastic region is the low-strain part of the curve
elastic = sample[sample["strain"] <= 0.012]

# fit a straight line: degree 1 -> (slope, intercept)
slope, intercept = np.polyfit(elastic["strain"], elastic["stress_MPa"], 1)

E_MPa = slope
print(f"Fitted slope (Young's modulus) = {E_MPa:.0f} MPa = {E_MPa/1000:.1f} GPa")

# Note: this is teaching data, so the number is not a realistic metal modulus
# (real metals are ~70-200 GPa). What matters here is the *method*: fit a line,
# read a physical quantity from its slope.

# plot the data, then the fitted line on top
plt.figure()
plt.plot(sample["strain"], sample["stress_MPa"], marker="o", markersize=3, label="data")
plt.plot(elastic["strain"], slope * elastic["strain"] + intercept,
         color="red", linewidth=2, label="linear fit")
plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title(f"Elastic-region fit for {sample_id}")
plt.legend()
plt.show()

### Exercise 4 — Fit another sample
Repeat the fit for `S01` (annealed). Is its modulus higher or lower than the quenched `S03`?
*Tip:* copy the cell above and change `sample_id`. You can also try a different cutoff than `0.012`.

In [ ]:
# Exercise 4 — SOLUTION

sample_id = "S01"
sample = df[df["sample_id"] == sample_id]

elastic = sample[sample["strain"] <= 0.012]
slope, intercept = np.polyfit(elastic["strain"], elastic["stress_MPa"], 1)

print(f"Young's modulus for {sample_id}: {slope:.0f} MPa = {slope/1000:.1f} GPa")

plt.figure()
plt.plot(sample["strain"], sample["stress_MPa"], marker="o", markersize=3, label="data")
plt.plot(elastic["strain"], slope * elastic["strain"] + intercept,
         color="red", linewidth=2, label="linear fit")
plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title(f"Elastic-region fit: {sample_id}")
plt.legend()
plt.show()

# The modulus of S01 (annealed) is similar to S03 (quenched) — heat treatment
# changes strength but not the stiffness (elastic modulus) in this teaching dataset.


## 7. A second dataset: hardness
Now a different measurement: **hardness** (HV), measured at several positions on each sample. This file has many repeated measurements per treatment, which is exactly what we need for distributions and error bars.
The column `is_valid` flags bad measurements (e.g. an indentation on a defect); we keep only the valid ones.

In [ ]:
hardness = load_table("hardness_positions.csv")
valid_hardness = hardness[hardness["is_valid"]]      # keep only the valid measurements

print("All measurements:", len(hardness), "| valid:", len(valid_hardness))
valid_hardness.head()

## 8. Histogram: distribution of hardness
A histogram shows how a single numerical variable is distributed.

In [ ]:
plt.figure()
plt.hist(valid_hardness["hardness_HV"], bins=10, edgecolor="black")
plt.xlabel("Hardness [HV]")
plt.ylabel("Count")
plt.title("Distribution of hardness values")
plt.show()

## 9. Box plot: hardness by treatment
A box plot compares the distribution of a value across groups. pandas can do it in one call.

In [ ]:
valid_hardness.boxplot(column="hardness_HV", by="treatment")
plt.xlabel("Treatment")
plt.ylabel("Hardness [HV]")
plt.title("Hardness by treatment")
plt.suptitle("")          # remove the automatic pandas title
plt.show()

## 10. Bar plot with error bars: how confident are we?
This is the scientist's question. We compute the **mean** and the **standard deviation** of hardness for each treatment, and draw the std as an error bar. If two bars overlap a lot within their error bars, the difference between them may not be meaningful.

In [ ]:
summary = (
    valid_hardness
    .groupby("treatment")["hardness_HV"]
    .agg(["mean", "std"])
    .reset_index()
)
summary

In [ ]:
plt.figure()
plt.bar(summary["treatment"], summary["mean"], yerr=summary["std"], capsize=5)
plt.xlabel("Treatment")
plt.ylabel("Mean hardness [HV]")
plt.title("Mean hardness by treatment (error bar = standard deviation)")
plt.show()

### Exercise 5 — Interpret
Looking at the bar plot: which treatment is hardest? Are the differences large compared with the error bars? Write two sentences.

In [ ]:
# Exercise 5 — SOLUTION (interpretation)

# Quenched is the hardest treatment — its bar sits clearly above all others.
# The differences ARE large relative to the error bars: the bars do not overlap,
# which suggests the treatment effect is real and not just measurement noise.


## 11. Saving a figure
For reports and slides you save figures to files with `plt.savefig(...)`.
> **Note:** we are in JupyterLite (Python in the browser), so the file is written to a temporary virtual disk and will not appear on your computer. In a normal Python install it would be saved next to your notebook.

In [ ]:
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

sample_id = "S03"
sample = df[df["sample_id"] == sample_id]

plt.figure()
plt.plot(sample["strain"], sample["stress_MPa"])
plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title(f"Stress-strain curve: {sample_id}")
plt.tight_layout()

output_path = FIG_DIR / "stress_strain_S03.png"
plt.savefig(output_path, dpi=150)
plt.show()
print("Saved figure to:", output_path)

## 12. Mini-project — a small analysis report
Put it together. Using the data above, produce:
1. one stress-strain curve for a sample of your choice;
2. one figure comparing all samples;
3. a fit of the elastic modulus for that sample;
4. a bar plot of mean hardness by treatment (with error bars);
5. two or three sentences of interpretation.

The core workflow:
```text
load data -> calculate quantities -> summarize -> plot/fit -> interpret
```

In [ ]:
# Mini-project — SOLUTION

# 1. One stress-strain curve
sample_id = "S03"
sample = df[df["sample_id"] == sample_id]

plt.figure()
plt.plot(sample["strain"], sample["stress_MPa"])
plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title(f"Stress-strain curve: {sample_id}")
plt.show()

# 2. All samples in one figure
plt.figure()
for sid in df["sample_id"].unique():
    s = df[df["sample_id"] == sid]
    plt.plot(s["strain"], s["stress_MPa"], label=sid)
plt.xlabel("Strain [-]")
plt.ylabel("Stress [MPa]")
plt.title("All samples")
plt.legend()
plt.show()

# 3. Elastic modulus for S03
elastic = sample[sample["strain"] <= 0.012]
slope, intercept = np.polyfit(elastic["strain"], elastic["stress_MPa"], 1)
print(f"Young's modulus (S03): {slope:.0f} MPa = {slope/1000:.1f} GPa")

# 4. Bar plot mean hardness by treatment
summary_h = (
    valid_hardness.groupby("treatment")["hardness_HV"]
    .agg(mean="mean", std="std")
    .reset_index()
)
plt.figure()
plt.bar(summary_h["treatment"], summary_h["mean"], yerr=summary_h["std"], capsize=5)
plt.xlabel("Treatment")
plt.ylabel("Mean hardness [HV]")
plt.title("Mean hardness by treatment (error bar = std)")
plt.show()

# 5. Interpretation
# S03 (quenched) is the strongest (~516 MPa) and hardest (~168 HV), but least ductile.
# S01 (annealed) is the softest (~115 HV) and most ductile (~0.29 strain at UTS).
# There is a clear strength-ductility trade-off across heat treatments.


## 13. Optional — the `fig, ax` plotting style
So far we used `plt.plot(...)` directly. A more explicit style creates a figure and an axis object first. It behaves the same for simple plots, but scales better to complex, multi-panel figures.

In [ ]:
sample_id = "S03"
sample = df[df["sample_id"] == sample_id]

fig, ax = plt.subplots()
ax.plot(sample["strain"], sample["stress_MPa"])
ax.set_xlabel("Strain [-]")
ax.set_ylabel("Stress [MPa]")
ax.set_title(f"Stress-strain curve (object style): {sample_id}")
plt.show()

## Recap / checklist
You can now:
- make line, scatter, histogram, bar (with error bars) and box plots;
- add labels, units, titles and legends;
- compute summary metrics per sample;
- **fit a straight line** and read a physical quantity (the elastic modulus) from the slope;
- compare groups and ask whether the differences are meaningful;
- save a figure.

The most important idea: **a plot or a fit is only useful once you interpret it.**